# Try out the NVIDIA AI-Q Blueprint!
This notebook deploys the NVIDIA AI-Q Blueprint — a reference application for an enterprise-grade research agent built with the [NeMo Agent toolkit](https://docs.nvidia.com/nemo/agent-toolkit/latest/). It uses a two-tier research architecture that keeps simple queries fast while reserving multi-phase deep research for complex topics. The blueprint demonstrates an AI-powered research workflow that combines intent-aware routing, shallow and deep research agents, and optional human-in-the-loop clarification, plus benchmarks and evaluation harnesses for evaluation-driven development.

Some of the key capabilities of this blueprint are:

<table>
  <thead>
    <tr><th>Capability</th><th>Description</th><th>Value</th></tr>
  </thead>
  <tbody>
    <tr><td>Intelligence Dynamic Routing</td><td>Selects the right model, workflow depth, and data sources</td><td>optimal cost & performance</td></tr>
    <tr><td>Custom Evaluation Metrics</td><td>Evaluation harness with transparent reasoning</td><td>trust, auditability, reliability</td></tr>
    <tr><td>Open Recipe</td><td>Fully open, modular codebase with pluggable components</td><td>easy to customize and extend</td></tr>
    <tr><td>Persistent Context Management</td><td>Virtual file system and workspaces for data, code, and insights</td><td>handles long-running workflows</td></tr>
  </tbody>
</table>

In this notebook, you will install the prerequisites, set up the workflow environment, and connect to NVIDIA-hosted NIM APIs. Once deployed, you will have a fully functional research assistant that can answer simple questions via fast tool-augmented lookup (shallow research) and produce citation-backed, publication-style reports for complex topics (deep research). You can run the blueprint with web search only, customize models and tools, and optionally add pluggable knowledge retrieval.


Note: this notebook is designed to run as a [brev.dev launchable](https://brev.nvidia.com/launchable/deploy?launchableID=env-3ArAhMZxDcUsqbBUL8SbE2fAbKp)

![Architecture diagram](https://github.com/NVIDIA-AI-Blueprints/aiq/raw/develop/docs/assets/AIQ-arch-light.png)

# Getting Started

> [Software Components](#software-components)  
> [Hardware Requirements](#hardware-requirements)  
> [Prerequisites](#prerequisites)  
> [Spin Up Blueprint](#spin-up-blueprint)  
> [Environment Setup](#environment-setup)  
> [Run agent with NAT](#run-this-agent-using-the-nemo-agent-toolkit-run-command)  
> [Meta / Shallow / Deep examples](#meta-responses)  
> [Deployment using Docker Compose](#deployment-using-docker-compose)  
> [Optional Guardrails and Sandbox Profiles](#optional-guardrails-and-sandbox-profiles)  
> [Next Steps](#next-steps)

## Software Components

The following are used by this project:

- [NVIDIA NeMo Agent toolkit](https://docs.nvidia.com/nemo/agent-toolkit/latest/)
- [NIM of nvidia/nemotron-3.5-lightning-30b-a3b](https://build.nvidia.com/nvidia/nemotron-3.5-lightning-30b-a3b/modelcard) for intent classification
- [NIM of nvidia/nemotron-3-ultra-550b-a55b](https://build.nvidia.com/nvidia/nemotron-3-ultra-550b-a55b) for shallow research and every deep-research role in this Brev launchable
- [NIM of nvidia/nemotron-3-embed-1b](https://build.nvidia.com/nvidia/nemotron-3-embed-1b) (Optional)
- [NIM of nvidia/nemotron-3-nano-omni-30b-a3b-reasoning](https://build.nvidia.com/nvidia/nemotron-3-nano-omni-30b-a3b-reasoning) (Optional)
- [NVIDIA google/gemma-4-31b-it](https://build.nvidia.com/google/gemma-4-31b-it) (Optional)
- [Tavily Search API](https://tavily.com/) for web search
- [NeMo Guardrails](https://docs.nvidia.com/nemo/guardrails/latest/) through NeMo Agent Toolkit middleware (Optional)
- [NVIDIA OpenShell](https://docs.nvidia.com/openshell/latest/about/installation) (Optional)

## Hardware Requirements

We have two deployment options: 

- **Using Hosted NIMs**:
If you are deploying this blueprint without the NVIDIA RAG knowledge layer backend, there are **no GPU requirements** for this blueprint on its own.

- **Self-hosting NIMs:**
See the model cards for current deployment options for `nvidia/nemotron-3.5-lightning-30b-a3b` and `nvidia/nemotron-3-ultra-550b-a55b`. This Brev launchable uses Ultra for shallow research because the NVIDIA API Catalog Lightning serving profile has a known citation-output reliability limitation. A validated self-hosted Lightning profile remains the lower-latency alternative.  
If using NVIDIA RAG knowledge layer backend, please see [NVIDIA RAG Documentation Pages](https://docs.nvidia.com/rag/latest/support-matrix.html)



## Prerequisites

- **Docker Compose**
- **LLM API key** – For your chosen provider (required for LLM inference):
    - **NVIDIA API Key** from [NVIDIA AI](https://build.nvidia.com/) (for NIM models)
Optional requirements:
- **Web search API key**:
  - **Tavily** – [tavily.com](https://tavily.com) (for general web search)
- **Node.js 22+ and npm** (optional, for web UI mode)

## Spin Up Blueprint

We will be using docker compose to deploy this blueprint and interact with the NAT `run` command as well as through the custom chat-based frontend UI.

### Environment Setup

1. **Clone** the NVIDIA AI-Q Blueprint workflow repository
>**Note**: Skip this step if you are not on brev and instead running from within the repository.

In [ ]:
%%bash
# Only run this cell in a brev launchable
git clone https://github.com/NVIDIA-AI-Blueprints/aiq.git

In [ ]:
# Only run this cell in a brev launchable
# moves the notebook into the directory
%cd /home/ubuntu/aiq
repo_root = "/home/ubuntu/aiq"

2. **Setup** Environment

In [ ]:
# Run setup from the project root (where pyproject.toml and scripts/setup.sh live).
import os
from pathlib import Path

cwd = Path.cwd().resolve()
configured_repo_root = globals().get("repo_root")
REPO_ROOT = Path(configured_repo_root).expanduser().resolve() if configured_repo_root else None

search_roots = [cwd, *cwd.parents]
if REPO_ROOT is not None:
    search_roots = [REPO_ROOT, *search_roots]

REPO_ROOT = None
for candidate in search_roots:
    if (candidate / "pyproject.toml").exists() and (candidate / "scripts" / "setup.sh").exists():
        REPO_ROOT = candidate
        break

if REPO_ROOT is None:
    raise FileNotFoundError("Could not locate project root containing pyproject.toml and scripts/setup.sh")

os.environ["REPO_ROOT"] = str(REPO_ROOT)
os.chdir(REPO_ROOT)
os.environ["UV_VENV_CLEAR"] = "1"
get_ipython().system("./scripts/setup.sh")

In [ ]:
%%bash
source .venv/bin/activate

3. **Obtain and set** API keys and environment variables

<table>
  <thead>
    <tr><th>API</th><th>Environment Variable</th><th>Purpose</th></tr>
  </thead>
  <tbody>
    <tr><td>NVIDIA</td><td><code>NVIDIA_API_KEY</code></td><td>NIM / NGC model inference (required)</td></tr>
    <tr><td>Tavily</td><td><code>TAVILY_API_KEY</code></td><td>Web search (recommended)</td></tr>
  </tbody>
</table>

**NVIDIA API key:**  
1. Go to [NVIDIA Build](https://build.nvidia.com/explore/discover) or [NGC](https://ngc.nvidia.com/).  
2. Sign in and open a model card (e.g. Nemotron).  
3. Use **Get API Key** → **Generate Key**, then copy the key (starts with `nvapi-`).  

**Tavily API key:**  
1. Go to [tavily.com](https://tavily.com) and create an account.  
2. Create an API key in the dashboard and add it to your environment.  

In [ ]:
import getpass
import os

if "NVIDIA_API_KEY" not in os.environ or os.environ["NVIDIA_API_KEY"] == "":
    nvidia_api_key = getpass.getpass("Enter your NVIDIA API key: ")
    os.environ["NVIDIA_API_KEY"] = nvidia_api_key

if "TAVILY_API_KEY" not in os.environ or os.environ["TAVILY_API_KEY"] == "":
    tavily_api_key = getpass.getpass("Enter your Tavily API key: ")
    os.environ["TAVILY_API_KEY"] = tavily_api_key

### Run this agent using the NeMo Agent Toolkit [`run` command](https://docs.nvidia.com/nemo/agent-toolkit/latest/run-workflows/about-running-workflows.html#using-the-nat-run-command)

In [ ]:
%%writefile config_simple_researcher.yml

general:
  telemetry:
    logging:
      console:
        _type: console
        level: INFO

llms:
  nemotron_lightning_intent_llm:
    _type: nim
    model_name: nvidia/nemotron-3.5-lightning-30b-a3b
    base_url: "https://integrate.api.nvidia.com/v1"
    api_key: ${NVIDIA_API_KEY}
    temperature: 0.1
    top_p: 0.9
    max_tokens: 1024
    num_retries: 5
    parallel_tool_calls: false
    chat_template_kwargs:
      enable_thinking: false

  # The API Catalog Lightning serving profile can intermittently produce citation-incomplete
  # shallow drafts. AI-Q fails closed rather than publishing them. For a lower-latency
  # alternative, self-host Lightning with a serving profile validated end to end with AI-Q.
  # nemotron_lightning_agent_llm:
  #   _type: nim
  #   model_name: nvidia/nemotron-3.5-lightning-30b-a3b
  #   base_url: "http://<your-lightning-endpoint>/v1"
  #   api_key: ${SELF_HOSTED_LIGHTNING_API_KEY}
  #   temperature: 0.2
  #   top_p: 0.7
  #   max_tokens: 8192
  #   num_retries: 5
  #   parallel_tool_calls: false
  #   chat_template_kwargs:
  #     enable_thinking: true

  nemotron_ultra_shallow_llm:
    _type: nim
    model_name: nvidia/nemotron-3-ultra-550b-a55b
    base_url: "https://integrate.api.nvidia.com/v1"
    api_key: ${NVIDIA_API_KEY}
    temperature: 0.2
    top_p: 0.7
    max_tokens: 8192
    num_retries: 5
    parallel_tool_calls: false
    chat_template_kwargs:
      enable_thinking: true

  nemotron_ultra_llm:
    _type: nim
    model_name: nvidia/nemotron-3-ultra-550b-a55b
    base_url: "https://integrate.api.nvidia.com/v1"
    api_key: ${NVIDIA_API_KEY}
    temperature: 0.2
    top_p: 0.7
    max_tokens: 16384
    num_retries: 5
    chat_template_kwargs:
      enable_thinking: false

  nemotron_ultra_writer_llm:
    _type: nim
    model_name: nvidia/nemotron-3-ultra-550b-a55b
    base_url: "https://integrate.api.nvidia.com/v1"
    api_key: ${NVIDIA_API_KEY}
    temperature: 0.2
    top_p: 0.7
    max_tokens: 32768
    num_retries: 5
    chat_template_kwargs:
      enable_thinking: false

functions:
  web_search_tool:
    _type: tavily_web_search
    max_results: 5
    max_retries: 3
    advanced_search: false
    max_content_length: 1000

  advanced_web_search_tool:
    _type: tavily_web_search
    max_results: 2
    advanced_search: true

  intent_classifier:
    _type: intent_classifier
    llm: nemotron_lightning_intent_llm
    tools:
      - web_search_tool

  shallow_research_agent:
    _type: shallow_research_agent
    llm: nemotron_ultra_shallow_llm
    tools:
      - web_search_tool
    max_llm_turns: 20
    max_tool_iterations: 5

  deep_research_agent:
    _type: deep_research_agent
    orchestrator_llm: nemotron_ultra_llm
    source_router_llm: nemotron_ultra_llm
    researcher_llm: nemotron_ultra_llm
    planner_llm: nemotron_ultra_llm
    writer_llm: nemotron_ultra_writer_llm
    tools:
      - advanced_web_search_tool

workflow:
  _type: chat_deepresearcher_agent
  interactive_auth: false
  checkpoint_db: ${AIQ_CHECKPOINT_DB:-./checkpoints.db}


**What this config does:** 

This config defines a NAT workflow for the AI-Q chat researcher. 

**`general`** includes the API worker settings and telemetry configuration.

**`general.front_end`** wires the Web API (AI-Q API Worker), async job store, and CORS for the chat UI.

**`llms`** keeps Nemotron 3.5 Lightning for intent classification and uses a bounded Nemotron Ultra profile for shallow research, plus role-specific Ultra profiles for deep research. The commented Lightning shallow profile is intended for a self-hosted endpoint validated with AI-Q; it is not configured to use NVIDIA API Catalog.

**`functions`** register tools (Tavily web search, plus advanced search), the intent classifier, and the shallow and deep research agents with their assigned tools and LLMs.
 
**`workflow`** is set to `chat_deepresearcher_agent`. See source code for this full agent in `src/aiq_agent/agents/chat_researcher/agent.py`.

**Optional settings:** 

`interactive_auth: false` disables interactive auth prompts for this local demo config.

**Main agents:** intent classifier, shallow research, deep research. 
**Tools:** web search tool for shallow researcher and advanced web search for deep research.

>**Optional — Frontier model config:** For improved research quality, you can use `configs/config_frontier_models.yml`, which assigns `gpt-5.6-luna` to intent classification, shallow research, source routing, and deep-research execution, and `gpt-5.6-sol` to clarification, orchestration, planning, and writing. When using Docker Compose or the web UI, set `BACKEND_CONFIG` to `/app/configs/config_frontier_models.yml`. Remember to set the required API keys in your `.env` file.


#### Meta Responses 

Documentation: `docs/source/architecture/agents/intent-classifier.md`

Meta responses handle greetings and questions about what AI-Q can do.


In [ ]:
!.venv/bin/nat run --config_file config_simple_researcher.yml --input "Hello, what can you do?" | sed 's/\\n/\n/g'

#### Shallow Research

Documentation: `docs/source/architecture/agents/shallow-researcher.md`

Shallow research handles focused factual questions with a short tool-assisted lookup.


In [ ]:
!.venv/bin/nat run --config_file config_simple_researcher.yml --input "What is NVIDIA Spectrum-X primarily designed for?" | sed 's/\\n/\n/g'

#### Deep Research 

Documentation: `docs/source/architecture/agents/deep-researcher.md`  

<div class="alert alert-block alert-info">
<b>Note:</b> Deep research can take several minutes: multiple LLM calls and web searches run in sequence. Watch the logs to see planning and research steps.
</div>


In [ ]:
!.venv/bin/nat run --config_file config_simple_researcher.yml --input "Conduct a comprehensive technical comparison between NVIDIA GB300 and Google Ironwood AI accelerator platforms. Generate a structured report with the following sections: Compute Architecture, Memory Subsystem, Interconnect & Scalability, System & Power, Software Ecosystem and Benchmarks" | sed 's/\\n/\n/g'

### Deployment using Docker Compose

1. Set environment variables for docker compose deployment

In [ ]:
os.environ["BACKEND_URL"] = "http://aiq-agent:8000"
os.environ["REQUIRE_AUTH"] = "false"
os.environ["BACKEND_CONFIG"] = (
    "/app/configs/config_web_default_llamaindex.yml"  # path to the NAT config file from inside the docker container
)

2. Create an env file with secrets

In [ ]:
%%bash
cat > deploy/.env <<EOF
APP_ENV=development
LOG_LEVEL=INFO

AIQ_DEV_ENV=cli

NVIDIA_API_KEY=${NVIDIA_API_KEY}
TAVILY_API_KEY=${TAVILY_API_KEY}
EOF

Or run `cp deploy/.env.example deploy/.env` and replace the placeholders in the created `.env` file

3. Deploy the containers

**Optional**: you can build the image and start up the containers in the terminal with the command below:

`docker compose -f deploy/compose/docker-compose.yaml up -d --build`

In [ ]:
# build the docker image
!docker compose -f deploy/compose/docker-compose.yaml build --quiet

In [ ]:
# start up the containers in detached mode
!docker compose -f deploy/compose/docker-compose.yaml up -d > /dev/null 2>&1

4. Verify the containers are running

In [ ]:
# Confirm all the below mentioned containers are running.
import subprocess

result = subprocess.run(
    ["docker", "ps", "--format", "table {{.ID}}\t{{.Names}}\t{{.Status}}"],
    check=False,
    capture_output=True,
    text=True,
)

print(result.stdout)

The output should look something like this:

```
CONTAINER ID   NAMES                    STATUS
5f5bf0e2438f   aiq-blueprint-ui         Up 2 minutes (healthy)
e11fad0dba6e   aiq-agent                Up 2 minutes
66d55656299b   aiq-postgres             Up 3 minutes (healthy)
```

At this point, you should be able to access the NVIDIA AI-Q frontend web application. On a local machine, visit http://localhost:3000. On Brev, open the forwarded/proxied port 3000 URL from the Brev environment.

>**Warning:** This demo sets `REQUIRE_AUTH=false`. Keep the local or Brev URL private and use it only in a trusted demo environment. Anyone with access to the URL can access jobs, reports, and, when artifact capture is enabled, generated files. Enable authentication before sharing the deployment.

>**Tip:** If you are running this notebook on brev, you will need to make the port for the AI-Q frontend accessible. On the settings page for your machine, navigate to "Using Ports", enter "3000", click "Expose Port", and then click "I accept".

**Select data sources:** In the UI, open the **data sources** panel (e.g. from the right side or connections icon). You will see available sources (e.g. Web Search, and optionally Paper Search, Knowledge Layer, or enterprise sources if configured). Toggle **on** the sources you want the agent to use for the next query. Web Search is usually enabled by default.

**Send a query:** Type your question or research request in the chat input and send it. The agent will use only the **enabled** data sources for that query. For deep research, the UI may show clarification steps; respond as prompted. Results and the final report will appear in the chat.

**Follow up on a report:** After a deep research report completes, you can ask follow-up questions or request a targeted rewrite from the completed report view. AI-Q keeps the original report as context and starts a follow-up job when more research or synthesis is needed.

### Optional Guardrails and Sandbox Profiles

The default UI deployment above uses `configs/config_web_default_llamaindex.yml`. AI-Q also ships focused profiles for optional capabilities:

- **Guardrails:** `configs/config_web_default_guardrails.yml` adds NeMo Guardrails checks at selected workflow and agent boundaries. Select it by setting `BACKEND_CONFIG=/app/configs/config_web_default_guardrails.yml` in `deploy/.env` before starting Docker Compose. See `docs/source/customization/guardrails.md`.
- **OpenShell sandbox and durable artifacts:** `configs/config_openshell.yml` enables policy-bound sandbox execution, skills, and capture of generated files for the UI **Files** tab. Before using this profile, install and start the OpenShell gateway, generate or provide the policy file, and set the `AIQ_OPENSHELL_*` environment variables described in `docs/source/deployment/openshell.md`.

These profiles are optional and are not required for the basic web-search workflow.

In [ ]:
# Bring docker containers down
!docker compose -f deploy/compose/docker-compose.yaml down -v

In [ ]:
# Optional: Reset the environment
# use only if you want to stop containers, remove volumes, and delete cached images
!docker compose -f deploy/compose/docker-compose.yaml down -v --rmi all

## (Optional) Add Observability with LangSmith

[LangSmith](https://smith.langchain.com/) lets you trace, monitor, and debug your AI-Q agent runs end-to-end.

**1. Sign up and get an API key**

Go to [smith.langchain.com](https://smith.langchain.com/), create an account, and generate an API key. Create a project to collect your traces.

**2. Set your LangSmith environment variables**

In [ ]:
import getpass
import os

if "LANGSMITH_API_KEY" not in os.environ or os.environ["LANGSMITH_API_KEY"] == "":
    langsmith_api_key = getpass.getpass("Enter your LangSmith API key: ")
    os.environ["LANGSMITH_API_KEY"] = langsmith_api_key

if "LANGSMITH_PROJECT" not in os.environ or os.environ["LANGSMITH_PROJECT"] == "":
    os.environ["LANGSMITH_PROJECT"] = input("Enter your LangSmith project name: ")

**3. Add LangSmith tracing to your config**

Run the cell below to automatically update `config_simple_researcher.yml` with LangSmith tracing:

In [ ]:
config_path = "config_simple_researcher.yml"

with open(config_path) as f:
    content = f.read()

if "tracing:" not in content:
    content = content.replace(
        "        level: INFO\n",
        "        level: INFO\n\n"
        "    tracing:\n"
        "      langsmith:\n"
        "        _type: langsmith\n"
        "        project: ${LANGSMITH_PROJECT}\n"
        "        api_key: ${LANGSMITH_API_KEY}\n",
        1,
    )
    with open(config_path, "w") as f:
        f.write(content)
    print(f"Tracing config added to {config_path}")
else:
    print("Tracing config already present — no changes made.")

**4. Run the agent and verify traces**

Re-run the agent with the updated config:

In [ ]:
!.venv/bin/nat run --config_file config_simple_researcher.yml --input "What is NVIDIA Spectrum-X?" | sed 's/\\n/\n/g'

After the run completes, go to [smith.langchain.com](https://smith.langchain.com/), open your project, and you should see a new trace for the query above. Click into it to explore the agent's steps, LLM calls, and tool usage.

## (Optional) Use a Partner LLM Endpoint

You can swap the LLM provider in `config_simple_researcher.yml` with any OpenAI-compatible API.

For example, to use [Together.ai](https://www.together.ai/) with a supported model — sign up at [together.ai](https://www.together.ai/), generate an API key, and update the relevant Lightning or Ultra LLM block. The pattern below shows the provider change for one block:

```yaml
llms:
  nemotron_ultra_llm:
    _type: openai
    model_name: ${TOGETHER_MODEL_NAME}
    base_url: "https://api.together.ai/v1"
    api_key: ${TOGETHER_API_KEY}
    temperature: 0.1
    top_p: 0.3
    max_tokens: 16384
    num_retries: 5
```

Key changes from the default NIM config:
- `_type: nim` → `_type: openai`
- `base_url` → partner endpoint URL
- `api_key` → your `TOGETHER_API_KEY`
- Remove `chat_template_kwargs: enable_thinking: true` (NIM-specific)

> More provider guides coming soon.

## Next Steps

You have now deployed the NVIDIA AI-Q Blueprint and interacted with the chat-based agent. To continue this journey, checkout the [Github repo](https://github.com/NVIDIA-AI-Blueprints/aiq).

**More notebooks**: We have walkthroughs of the deep researcher agent and its customization in the `docs/notebooks/` directory.

**Customization Guide**: Check out the [customization guide](https://docs.nvidia.com/aiq-blueprint/latest/customization/index.html).